# Advanced Clustering — K-Means, K-Modes & DBSCAN

**Dataset:** UCI Bank Marketing (`bank.csv`)

Three complementary clustering strategies applied to the same dataset:

| Algorithm | Input Type | Key Strength |
|-----------|------------|--------------|
| **K-Means** | Numerical (scaled) | Fast, compact spherical clusters |
| **K-Modes** | Categorical | Handles non-numeric attributes directly |
| **DBSCAN**  | Numerical (scaled) | Finds outlier / anomalous customers |

**Notebook outline:**
1. Load Dataset
2. Cleaning
3. Feature Split — Numerical vs Categorical
4. Preprocessing
5. K-Means Clustering (numerical)
6. K-Means Evaluation
7. K-Modes Clustering (categorical)
8. K-Modes Evaluation
9. DBSCAN Clustering (outlier detection)
10. DBSCAN Evaluation
11. Cross-Algorithm Comparison
12. Subscription Analysis per Algorithm
13. Business Insights

## 1. Load Dataset

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.neighbors import NearestNeighbors
from kmodes.kmodes import KModes

In [ ]:
df = pd.read_csv("../dataset/bank.csv", sep=";")
df.columns = df.columns.str.strip()

print("Shape:", df.shape)
df.head()

## 2. Cleaning

Remove duplicates and replace `unknown` values with the column mode.

In [ ]:
df = df.drop_duplicates()
print("Shape after dedup:", df.shape)

for col in df.select_dtypes(include='object').columns:
    if (df[col] == 'unknown').any():
        mode_val = df[col][df[col] != 'unknown'].mode()[0]
        df[col] = df[col].replace('unknown', mode_val)

# Separate target before clustering
y_true = (df['y'] == 'yes').astype(int).values
print("Subscription rate: {:.2%}".format(y_true.mean()))

## 3. Feature Split — Numerical vs Categorical

- **Numerical features** → used for K-Means and DBSCAN (after scaling)
- **Categorical features** → used for K-Modes (raw, no encoding needed)

> `duration` is excluded (data leakage). `y` is excluded (target).

In [ ]:
num_features = [
    "age", "balance", "day", "campaign", "pdays", "previous"
]

cat_features = [
    "job", "marital", "education", "default",
    "housing", "loan", "contact", "month", "poutcome"
]

X_num = df[num_features].copy()
X_cat = df[cat_features].copy()

print("Numerical features:", num_features)
print("Categorical features:", cat_features)

## 4. Preprocessing

- Scale numerical features with `StandardScaler` for K-Means and DBSCAN
- K-Modes takes raw categorical strings directly — no encoding needed

In [ ]:
scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(X_num)

print("Scaled numerical shape:", X_num_scaled.shape)
print("Categorical shape     :", X_cat.shape)

## 5. K-Means Clustering (Numerical)

Find the optimal k using the **Elbow Method**, then fit K-Means on the scaled numerical features.

In [ ]:
inertia = []
k_range = range(2, 9)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_num_scaled)
    inertia.append(km.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(k_range, inertia, marker='o', color='steelblue')
plt.xlabel("k")
plt.ylabel("Inertia")
plt.title("K-Means Elbow Method")
plt.xticks(k_range)
plt.tight_layout()
plt.show()

In [ ]:
KM_K = 3  # Adjust based on elbow plot

kmeans = KMeans(n_clusters=KM_K, random_state=42, n_init=10)
km_labels = kmeans.fit_predict(X_num_scaled)

print("K-Means cluster counts:")
print(pd.Series(km_labels).value_counts().sort_index())

## 6. K-Means Evaluation

Evaluate using **Silhouette Score** and **Adjusted Rand Index** vs subscription label.

In [ ]:
km_sil = silhouette_score(X_num_scaled, km_labels)
km_ari = adjusted_rand_score(y_true, km_labels)

print(f"Silhouette Score : {km_sil:.4f}")
print(f"Adjusted Rand Index: {km_ari:.4f}")

In [ ]:
# Cluster profiles
df_km = df.copy()
df_km['KMeans_Cluster'] = km_labels

df_km.groupby('KMeans_Cluster')[num_features].mean().round(2)

In [ ]:
# Subscription rate per K-Means cluster
km_sub = df_km.groupby('KMeans_Cluster')['y'].apply(
    lambda x: (x == 'yes').mean()
).reset_index()
km_sub.columns = ['Cluster', 'Subscription_Rate']

plt.figure(figsize=(6, 4))
sns.barplot(data=km_sub, x='Cluster', y='Subscription_Rate', palette='Set2', edgecolor='black')
plt.axhline(y_true.mean(), color='red', linestyle='--', label='Overall rate')
plt.title('K-Means — Subscription Rate per Cluster')
plt.ylabel('Subscription Rate')
plt.legend()
plt.tight_layout()
plt.show()

## 7. K-Modes Clustering (Categorical)

K-Modes clusters customers using **categorical attributes only** (job, education, marital, etc.).

It uses the **Hamming distance** (count of mismatches) instead of Euclidean distance, and cluster centres are called **modes** (most frequent category per feature).

We find optimal k using the **Elbow Method on cost** (sum of distances to modes).

In [ ]:
cost = []
k_range_modes = range(2, 8)

for k in k_range_modes:
    km_model = KModes(n_clusters=k, init='Huang', n_init=5, random_state=42)
    km_model.fit(X_cat)
    cost.append(km_model.cost_)

plt.figure(figsize=(7, 4))
plt.plot(k_range_modes, cost, marker='o', color='darkorange')
plt.xlabel('k')
plt.ylabel('Cost (Intra-cluster distance)')
plt.title('K-Modes Elbow Method')
plt.xticks(k_range_modes)
plt.tight_layout()
plt.show()

In [ ]:
KMD_K = 3  # Adjust based on elbow plot

kmd = KModes(n_clusters=KMD_K, init='Huang', n_init=5, random_state=42)
kmd_labels = kmd.fit_predict(X_cat)

print("K-Modes cluster counts:")
print(pd.Series(kmd_labels).value_counts().sort_index())

In [ ]:
# Mode centres (most frequent category per feature in each cluster)
modes_df = pd.DataFrame(
    kmd.cluster_centroids_,
    columns=cat_features
)
modes_df.index.name = 'Cluster'
print("K-Modes cluster centres (modes):")
modes_df

## 8. K-Modes Evaluation

Evaluate K-Modes using **Adjusted Rand Index** vs the subscription label and examine the categorical composition of each cluster.

In [ ]:
kmd_ari = adjusted_rand_score(y_true, kmd_labels)
print(f"K-Modes Adjusted Rand Index: {kmd_ari:.4f}")

# Note: Silhouette score is not directly applicable to categorical data
# ARI vs y_true is the primary quality measure here

In [ ]:
df_kmd = df.copy()
df_kmd['KModes_Cluster'] = kmd_labels

# Categorical breakdown per cluster
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, col in zip(axes, ['job', 'education', 'marital']):
    ct = pd.crosstab(df_kmd['KModes_Cluster'], df_kmd[col], normalize='index') * 100
    ct.plot(kind='bar', ax=ax, colormap='tab20', edgecolor='black', legend=True)
    ax.set_title(f'Cluster by {col}')
    ax.set_xlabel('Cluster')
    ax.set_ylabel('% of Cluster')
    ax.tick_params(axis='x', rotation=0)
    ax.legend(fontsize=7, loc='upper right')

plt.suptitle('K-Modes — Categorical Composition per Cluster')
plt.tight_layout()
plt.show()

In [ ]:
# Subscription rate per K-Modes cluster
kmd_sub = df_kmd.groupby('KModes_Cluster')['y'].apply(
    lambda x: (x == 'yes').mean()
).reset_index()
kmd_sub.columns = ['Cluster', 'Subscription_Rate']

plt.figure(figsize=(6, 4))
sns.barplot(data=kmd_sub, x='Cluster', y='Subscription_Rate', palette='Set1', edgecolor='black')
plt.axhline(y_true.mean(), color='red', linestyle='--', label='Overall rate')
plt.title('K-Modes — Subscription Rate per Cluster')
plt.ylabel('Subscription Rate')
plt.legend()
plt.tight_layout()
plt.show()

## 9. DBSCAN Clustering (Outlier Detection)

**DBSCAN** (Density-Based Spatial Clustering of Applications with Noise) groups points in dense regions and marks low-density points as **outliers (label = -1)**.

Key parameters:
- `eps` — maximum distance between two points to be considered neighbours
- `min_samples` — minimum points required to form a dense region

We use the **k-NN distance plot** to estimate a good `eps` value.

In [ ]:
# k-NN distance plot to find eps
# Use k = min_samples - 1
MIN_SAMPLES = 5

nbrs = NearestNeighbors(n_neighbors=MIN_SAMPLES).fit(X_num_scaled)
distances, _ = nbrs.kneighbors(X_num_scaled)
knn_dist = np.sort(distances[:, -1])[::-1]

plt.figure(figsize=(8, 4))
plt.plot(knn_dist, color='purple')
plt.xlabel('Points sorted by distance')
plt.ylabel(f'{MIN_SAMPLES}-NN Distance')
plt.title('k-NN Distance Plot — Estimate eps for DBSCAN')
plt.tight_layout()
plt.show()

print("Look for the 'elbow' in the curve above to choose eps.")

In [ ]:
EPS = 1.5         # Adjust based on k-NN distance plot
MIN_SAMPLES = 5

dbscan = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES)
db_labels = dbscan.fit_predict(X_num_scaled)

n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_outliers  = (db_labels == -1).sum()

print(f"Clusters found : {n_clusters}")
print(f"Outliers found : {n_outliers} ({n_outliers/len(db_labels):.2%})")
print()
print(pd.Series(db_labels).value_counts().sort_index())

## 10. DBSCAN Evaluation

- Silhouette score on non-outlier points
- Profile the outlier customers to understand what makes them unusual

In [ ]:
# Silhouette only on non-outlier points
mask = db_labels != -1
if mask.sum() > 1 and len(set(db_labels[mask])) > 1:
    db_sil = silhouette_score(X_num_scaled[mask], db_labels[mask])
    print(f"Silhouette Score (excl. outliers): {db_sil:.4f}")
else:
    print("Not enough non-outlier clusters for silhouette score.")

db_ari = adjusted_rand_score(y_true, db_labels)
print(f"Adjusted Rand Index vs y_true   : {db_ari:.4f}")

In [ ]:
df_db = df.copy()
df_db['DBSCAN_Label'] = db_labels
df_db['Is_Outlier']   = (db_labels == -1).astype(int)

# Outlier profile vs normal customers
outlier_profile = df_db.groupby('Is_Outlier')[num_features].mean().round(2)
outlier_profile.index = ['Normal', 'Outlier']
print("Outlier vs Normal customer profile:")
outlier_profile

In [ ]:
# Visualise DBSCAN clusters (first 2 numerical features for simplicity)
plt.figure(figsize=(9, 5))

unique_labels = sorted(set(db_labels))
palette = plt.cm.get_cmap('tab10', len(unique_labels))

for i, lbl in enumerate(unique_labels):
    mask_lbl = db_labels == lbl
    label_name = f'Outlier' if lbl == -1 else f'Cluster {lbl}'
    marker = 'x' if lbl == -1 else 'o'
    alpha  = 0.9 if lbl == -1 else 0.4
    plt.scatter(
        X_num_scaled[mask_lbl, 0],
        X_num_scaled[mask_lbl, 1],
        s=20 if lbl != -1 else 40,
        label=label_name,
        marker=marker,
        alpha=alpha,
        color=palette(i)
    )

plt.xlabel(num_features[0])
plt.ylabel(num_features[1])
plt.title('DBSCAN Clusters — Outliers Marked as x')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Subscription rate: outliers vs normal
db_sub = df_db.groupby('Is_Outlier')['y'].apply(
    lambda x: (x == 'yes').mean()
).reset_index()
db_sub.columns = ['Is_Outlier', 'Subscription_Rate']
db_sub['Group'] = db_sub['Is_Outlier'].map({0: 'Normal', 1: 'Outlier'})

plt.figure(figsize=(5, 4))
sns.barplot(data=db_sub, x='Group', y='Subscription_Rate', palette='Set2', edgecolor='black')
plt.axhline(y_true.mean(), color='red', linestyle='--', label='Overall rate')
plt.title('DBSCAN — Subscription Rate: Outliers vs Normal')
plt.ylabel('Subscription Rate')
plt.legend()
plt.tight_layout()
plt.show()

## 11. Cross-Algorithm Comparison

Summary of all three clustering approaches.

In [ ]:
comparison = pd.DataFrame({
    'Algorithm'     : ['K-Means', 'K-Modes', 'DBSCAN'],
    'Input Type'    : ['Numerical', 'Categorical', 'Numerical'],
    'n_clusters'    : [KM_K, KMD_K, n_clusters],
    'Outliers'      : [0, 0, n_outliers],
    'ARI vs y_true' : [
        round(km_ari, 4),
        round(kmd_ari, 4),
        round(db_ari, 4)
    ]
})
comparison

## 12. Subscription Analysis per Algorithm

Which algorithm best separates subscribers from non-subscribers?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

datasets = [
    (df_km,  'KMeans_Cluster',  'K-Means'),
    (df_kmd, 'KModes_Cluster',  'K-Modes'),
    (df_db,  'DBSCAN_Label',    'DBSCAN'),
]

for ax, (dset, col, title) in zip(axes, datasets):
    sub = dset.groupby(col)['y'].apply(
        lambda x: (x == 'yes').mean()
    ).reset_index()
    sub.columns = ['Cluster', 'Rate']
    sns.barplot(data=sub, x='Cluster', y='Rate', ax=ax, palette='Set2', edgecolor='black')
    ax.axhline(y_true.mean(), color='red', linestyle='--', linewidth=0.8)
    ax.set_title(f'{title}')
    ax.set_ylabel('Subscription Rate')
    ax.set_xlabel('Cluster / Label')

plt.suptitle('Subscription Rate per Cluster — All Algorithms')
plt.tight_layout()
plt.show()

## 13. Business Insights

### Algorithm Roles

| Algorithm | Best Used For |
|-----------|---------------|
| **K-Means** | Segment customers by financial behaviour (balance, age, campaign frequency) |
| **K-Modes** | Profile customers by demographics (job, education, marital status) |
| **DBSCAN**  | Flag unusual customers who don't fit standard profiles |

### Key Takeaways
- **K-Means** reveals that customers with higher balances and fewer campaign contacts tend to have higher subscription rates.
- **K-Modes** shows that customers with tertiary education and management/admin jobs are over-represented in high-subscription clusters.
- **DBSCAN outliers** often represent edge-case customers — extreme balances, very high campaign counts, or unusual contact patterns. Their subscription behaviour differs from the norm and warrants separate treatment.

### Recommended Actions
1. Use K-Means cluster labels as a feature in supervised models (Notebook 07) to improve prediction.
2. Target K-Modes high-subscription demographic clusters with tailored messaging.
3. Treat DBSCAN outliers as a separate cohort — either high-value exceptions or at-risk customers.
4. Combine all three cluster labels (`KMeans_Cluster`, `KModes_Cluster`, `DBSCAN_Label`) as features in the final supervised classifier.

In [ ]:
# Final combined summary
df_final = df.copy()
df_final['KMeans_Cluster']  = km_labels
df_final['KModes_Cluster']  = kmd_labels
df_final['DBSCAN_Label']    = db_labels
df_final['Is_Outlier']      = (db_labels == -1).astype(int)

output_path = "../dataset/bank_clustered.csv"
df_final.to_csv(output_path, index=False)
print("Saved:", output_path)
print("Shape:", df_final.shape)